# Fooocus Easy Colab (run-all)

1) Set Colab runtime to GPU.
2) Run all cells.
3) Use the printed link `✅ Open this URL manually`.
4) If public URL does not open, use `✅ Fallback URL (Colab proxy)`.


In [ ]:
!pip install -q pygit2==1.15.1

%cd /content
!if [ ! -d "/content/Fooocus/.git" ]; then git clone -b cursor/git-eaf0 https://github.com/sunsideaspect/foocus_new.git Fooocus; fi
%cd /content/Fooocus
!git fetch origin cursor/git-eaf0
!git checkout cursor/git-eaf0
!git pull origin cursor/git-eaf0

%env TORCH_COMMAND=pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

import torch
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda, '| cuda_available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime is required. Colab: Runtime -> Change runtime type -> GPU, then rerun.')


In [ ]:
import re
import subprocess
import sys
from IPython.display import HTML, display

%cd /content/Fooocus

cmd = [
    sys.executable,
    "entry_with_update.py",
    "--preset", "realistic_identity",
    "--share",
    "--always-normal-vram",
    "--disable-in-browser"
]
print("Launching:", " ".join(cmd))

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

share_re = re.compile(r"https://[a-z0-9-]+\.gradio\.live")
local_re = re.compile(r"Running on local URL:\s+(http://127\.0\.0\.1:(\d+))")
printed_public = False
printed_proxy = False

for line in proc.stdout:
    print(line, end="")

    if (not printed_public):
        m = share_re.search(line)
        if m:
            share_url = m.group(0)
            print("\n✅ Open this URL manually:\n" + share_url + "\n")
            display(HTML(f'<a href="{share_url}" target="_blank">Open Gradio public URL</a>'))
            printed_public = True

    if (not printed_proxy):
        lm = local_re.search(line)
        if lm:
            port = lm.group(2)
            try:
                from google.colab import output
                proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
                if proxy_url:
                    print("\n✅ Fallback URL (Colab proxy):\n" + proxy_url + "\n")
                    display(HTML(f'<a href="{proxy_url}" target="_blank">Open Colab proxy URL</a>'))
                    printed_proxy = True
            except Exception as e:
                print(f"[Info] Could not create Colab proxy URL: {e}")
